# 04 — Cluster annotation

Equivalent to `scripts/04_annotate_clusters.py`. Produces **Figure 3**.

A cluster number means nothing. "Cluster 14 responds strongly to cocaine" is not a
finding; "Kenyon cells respond strongly to cocaine" is. This step turns numbers
into biology, and it requires **your judgement**, not a function call.

## The warning that matters most

The paper's C11 is Kenyon cells. **Your** cluster 11 is almost certainly something
else — Leiden numbers clusters by size, and your clustering is not theirs. Match by
**marker genes**, never by number. Writing "our C11 agrees with their C11" without
checking markers is the error most likely to be caught in peer review.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import anndata as ad

import config

sc.settings.verbosity = 3
sc.settings.figdir = config.FIG_DIR
sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=False)
sc.logging.print_header()

In [ ]:
adata = sc.read_h5ad(config.H5AD_CLUSTERED)
print(f'{adata.n_obs:,} cells, {adata.obs[config.LEIDEN_KEY].nunique()} clusters')

## Rank marker genes per cluster

`use_raw=True` is essential: `adata.X` holds scaled z-scores restricted to 2,000
HVGs. `adata.raw` holds log-normalized counts for all genes.

In [ ]:
sc.tl.rank_genes_groups(adata, groupby=config.LEIDEN_KEY,
                        method=config.DE_METHOD, use_raw=True, pts=True)

markers = sc.get.rank_genes_groups_df(adata, group=None)
markers.to_csv(config.TABLE_DIR / 'cluster_markers_all.csv', index=False)
markers.head()

In [ ]:
for cl in sorted(adata.obs[config.LEIDEN_KEY].unique(), key=int):
    top = markers[markers['group'] == cl].nlargest(5, 'scores')['names'].tolist()
    n = int((adata.obs[config.LEIDEN_KEY] == cl).sum())
    print(f'{cl:>3} (n={n:>6,}): ' + ', '.join(top))

## Canonical marker panel

Read this row by row:

- `repo` high + `elav` low → **glia**
- `elav` / `nSyb` / `brp` high → **neuron**
- then subdivide neurons by neurotransmitter: `VAChT`, `Gad1`, `VGlut`, `ple`, `SerT`, `Tdc2`
- `ey` + `Fas2` + `rut` → **Kenyon cells** (the mushroom body, the cocaine-relevant population)

**Verify every marker against FlyBase before citing it.** Marker lists are a known
AI hallucination risk, and the course policy puts validation on you.

In [ ]:
var_groups = {}
for cell_type, genes in config.CANONICAL_MARKERS.items():
    present = [g for g in genes if g in adata.raw.var_names]
    if present:
        var_groups[cell_type] = present

missing = [g for gs in config.CANONICAL_MARKERS.values() for g in gs
           if g not in adata.raw.var_names]
print('not found (check FlyBase spelling):', missing)

In [ ]:
sc.pl.dotplot(adata, var_names=var_groups, groupby=config.LEIDEN_KEY,
              use_raw=True, standard_scale='var', dendrogram=True)

In [ ]:
sc.pl.matrixplot(adata, var_names=var_groups, groupby=config.LEIDEN_KEY,
                 use_raw=True, standard_scale='var', dendrogram=True, cmap='viridis')

In [ ]:
key = [g for g in ['repo','elav','nSyb','ey','Fas2','VAChT','Gad1',
                   'VGlut','ple','SerT','Tdc2','ninaE'] if g in adata.raw.var_names]
sc.pl.umap(adata, color=key, ncols=4, use_raw=True)

## Exact numbers, for the ambiguous clusters

Reading values beats squinting at dot sizes when two clusters look similar.

In [ ]:
flat = [g for gs in var_groups.values() for g in gs]
expr = sc.get.obs_df(adata, keys=flat, use_raw=True)
expr[config.LEIDEN_KEY] = adata.obs[config.LEIDEN_KEY].values
mean_expr = expr.groupby(config.LEIDEN_KEY, observed=True).mean().round(3)
mean_expr.to_csv(config.TABLE_DIR / 'marker_mean_expression_by_cluster.csv')
mean_expr

## Inspect one gene at a time

Change `gene` and re-run while you work through the worksheet.

In [ ]:
gene = 'ey'   # try: repo, alrm, moody, elav, Fas2, rut, VAChT, Gad1, VGlut, ple

if gene in adata.raw.var_names:
    sc.pl.umap(adata, color=gene, use_raw=True)
    e = sc.get.obs_df(adata, keys=[gene], use_raw=True)
    e[config.LEIDEN_KEY] = adata.obs[config.LEIDEN_KEY].values
    display(e.groupby(config.LEIDEN_KEY, observed=True).mean()
             .sort_values(gene, ascending=False).head(8))
else:
    print(f'{gene} not found')

## The worksheet

Deliberately blank. Filling it in **is** the intellectual work of this step, and you
must be able to defend every label at grading.

In [ ]:
counts = adata.obs[config.LEIDEN_KEY].value_counts().sort_index()
worksheet = pd.DataFrame({
    'cluster': counts.index,
    'n_cells': counts.values,
    'pct_of_total': (100 * counts.values / adata.n_obs).round(2),
    'top_markers': [', '.join(markers[markers['group']==cl].nlargest(5,'scores')['names'])
                    for cl in counts.index],
    'cell_type_ANNOTATE_ME': '',
    'paper_cluster_match': '',
    'evidence_notes': '',
})
worksheet.to_csv(config.TABLE_DIR / 'annotation_worksheet.csv', index=False)
worksheet

**Now:** fill in `cell_type_ANNOTATE_ME` using the dotplots above and Supplemental
Table S4 of the paper. Save as `annotation_worksheet_FILLED.csv` in the same folder,
then run the next cell.

In [ ]:
filled = config.TABLE_DIR / 'annotation_worksheet_FILLED.csv'

if filled.exists():
    ann = pd.read_csv(filled, dtype={'cluster': str})
    mapping = dict(zip(ann['cluster'], ann['cell_type_ANNOTATE_ME']))
    adata.obs['cell_type'] = (adata.obs[config.LEIDEN_KEY].astype(str)
                              .map(mapping).fillna('Unannotated').astype('category'))
    sc.pl.umap(adata, color='cell_type', legend_loc='on data', legend_fontsize=5)
else:
    adata.obs['cell_type'] = 'Unannotated'
    print('Worksheet not filled in yet — saving with placeholder labels.')

adata.write(config.H5AD_ANNOTATED)
print('Wrote', config.H5AD_ANNOTATED)